# Coachly NLU — Fine Tuning
**Prima di tutto:** `Runtime → Change runtime type → T4 GPU`

Esegui le celle in ordine con `Shift+Enter`.

In [ ]:
# Cella 1 — Monta Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cella 2 — Copia il progetto da Drive
import os, shutil

DRIVE_SRC = '/content/drive/MyDrive/voice-ml-recognizer'
LOCAL_DIR = '/content/proj'

if os.path.exists(LOCAL_DIR):
    shutil.rmtree(LOCAL_DIR)
shutil.copytree(DRIVE_SRC, LOCAL_DIR)
os.chdir(LOCAL_DIR)
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Cella 3 — Installa dipendenze
# pytorch-crf: CRF layer per sequenze BIO valide
# seqeval: metriche NER (F1 per entity type)
# onnx/onnxruntime: export modello per produzione
import subprocess, sys

pkgs = [
    'transformers>=4.38.0',
    'seqeval>=1.2.2',
    'onnx>=1.15.0',
    'onnxruntime>=1.17.0',
    'pytorch-crf>=0.7.2',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    check=True
)
print('Dipendenze ok.')

# Verifica import critici
import torch
from torchcrf import CRF
from transformers import XLMRobertaTokenizerFast
from seqeval.metrics import f1_score
import onnx
print(f'torch      : {torch.__version__}')
print(f'torchcrf   : ok')
print(f'transformers: ok')
print(f'seqeval    : ok')
print(f'onnx       : {onnx.__version__}')
print(f'GPU disponibile: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cella 4 — Genera dataset
# Se il dataset esiste già su Drive (resume), salta questo step
import os
if os.path.exists('data/train.json'):
    import json
    with open('data/train.json') as f:
        n = len(json.load(f))
    print(f'Dataset già presente: {n} esempi train — skip generazione.')
else:
    print('Generazione dataset...')
    import subprocess, sys
    subprocess.run([sys.executable, 'generate-dataset.py'], check=True)

In [ ]:
# Cella 5 — Fine tuning + export ONNX + demo inference
# Durata stimata su T4: 20-30 minuti (20 epoch, batch effettivo 64)
# Se esiste un resume checkpoint su Drive, riparte dall'epoch salvata.
import subprocess, sys
result = subprocess.run([sys.executable, 'colab_train.py'], check=False)
if result.returncode != 0:
    print('\n[ERRORE] colab_train.py uscito con errore. Controlla i log sopra.')

In [ ]:
# Cella 6 — Verifica risultati
import json, os

results_path = '/content/proj/output/workout_nlu/results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        r = json.load(f)
    print('=== Risultati finali ===')
    print(f"Best val combined : {r['best_val']:.4f}")
    print(f"Test intent_acc   : {r['test']['intent_acc']:.4f}")
    print(f"Test slot_f1      : {r['test']['slot_f1']:.4f}")
    print(f"Test combined     : {r['test']['combined']:.4f}")
else:
    print('results.json non trovato — il training potrebbe non essere terminato.')

# Lista file prodotti
out_dir = '/content/proj/output/workout_nlu'
if os.path.exists(out_dir):
    print('\nFile prodotti:')
    for f in sorted(os.listdir(out_dir)):
        size = os.path.getsize(os.path.join(out_dir, f))
        print(f'  {f:30s} {size/1e6:.1f} MB')

In [ ]:
# Cella 7 — Salva tutto su Drive
import os, shutil

DRIVE_OUT = '/content/drive/MyDrive/coachly_nlu'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree('/content/proj/output/workout_nlu', DRIVE_OUT, dirs_exist_ok=True)
print('Salvato su Drive:', DRIVE_OUT)
print('File:', os.listdir(DRIVE_OUT))